# 02 — Build an inspectable analysis assistant

Separate the chooser from the scientific computation. You will inspect a fixed numerical recipe and replace the chooser without changing tool permissions. The default path makes no model calls; a later optional cell can connect a separately installed local model.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, sys, urllib.request

# Replaced during publication with the commit containing the reviewed helper.
REF = "faf63f0f09cba07291e8c3d09ef95974b8404d77"
candidates = [Path("agent_studio.py"), Path("../agent_studio.py")]
source_path = next((p.resolve() for p in candidates if p.is_file()), None)
origin = "local copy"
if source_path is None:
    origin = f"GitHub commit {REF}"
    url = ("https://raw.githubusercontent.com/ZagElSayed/"
           f"Introduction-to-Neuroinformatics/{REF}/"
           "chapters/ch19-ai-agents/agent_studio.py")
    payload = urllib.request.urlopen(url, timeout=30).read()
    source_path = Path("agent_studio.py").resolve()
    source_path.write_bytes(payload)
manifest = {"origin": origin, "source_path": str(source_path),
            "sha256": hashlib.sha256(source_path.read_bytes()).hexdigest()}
Path("agent_source_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
spec = importlib.util.spec_from_file_location("agent_studio", source_path)
studio_module = importlib.util.module_from_spec(spec)
sys.modules["agent_studio"] = studio_module
spec.loader.exec_module(studio_module)
Studio = studio_module.Studio

## Inspect the recipe and source

The default fixture has 256 samples at 128 Hz. The direct DFT uses a symmetric Hann window and density normalization. Alpha power sums bins from 8 through 12 Hz inclusive and multiplies by 0.5 Hz. The energy check compares the full integrated spectrum with window-weighted time-domain energy. This is not a clinical biomarker validation.

In [ ]:
import inspect
print(json.dumps(studio_module.RECIPE, indent=2))
print(inspect.getsource(studio_module.alpha_periodogram))
studio = Studio("notebook02_outputs")
run = studio.run("toy_good")
print(json.dumps(run["values"].get("compute_alpha"), indent=2))

## Replace the chooser

The default chooser follows the first available tool. The next chooser fetches the recipe first whenever it is available. Both must satisfy exactly the same runtime prerequisites. Compare their actual trajectories. A different order alone is not a performance improvement.

In [ ]:
def recipe_first(view):
    allowed = view["allowed_tools"]
    name = "read_recipe" if "read_recipe" in allowed else allowed[0]
    return {"tool": name, "arguments": {"dataset_id": view["dataset_id"]}}

alternate = studio.run("toy_good", proposer=recipe_first)
for label, result in [("baseline", run), ("recipe first", alternate)]:
    print(label, result["status"],
          [e.get("proposal", {}).get("tool") for e in result["trajectory"]])

## A proposal cannot expand the tool set

This deliberately invalid proposer attempts an unregistered delete tool. Inspect the denied proposal and verify that there was no matching executed tool. This demonstrates this allow-list check, not a general prompt-injection defense.

In [ ]:
def invalid_proposal(view):
    return {"tool": "delete", "arguments": {"dataset_id": view["dataset_id"]}}

blocked = studio.run("toy_good", proposer=invalid_proposal)
print(blocked["status"], blocked["reason"])
print(json.dumps(blocked["trajectory"], indent=2))

## Optional local LLM planner

After the baseline works, install a local Ollama server and an appropriate local model using the official resources. This cell is disabled by default and downloads no model. Use local Jupyter: hosted Colab's loopback address is its remote kernel, not your laptop. The adapter sends only the synthetic task state, parses a schema-shaped proposal, and passes it to the same runtime checks. It does not give the model source-storage credentials. Record the local server version and installed model digest separately, and retain the request/response trace. Success is not assumed; compare repeated held-out tasks with the baseline.


In [ ]:
ENABLE_LLM = False
MODEL = ""  # Exact name of a separately installed local model.
if ENABLE_LLM:
    adapter_candidates = [source_path.parent / "ollama_planner.py"]
    adapter_path = next((p for p in adapter_candidates if p.is_file()), None)
    if adapter_path is None:
        url = ("https://raw.githubusercontent.com/ZagElSayed/"
               f"Introduction-to-Neuroinformatics/{REF}/"
               "chapters/ch19-ai-agents/ollama_planner.py")
        adapter_path = Path("ollama_planner.py").resolve()
        adapter_path.write_bytes(urllib.request.urlopen(url, timeout=30).read())
    adapter_spec = importlib.util.spec_from_file_location("ollama_planner", adapter_path)
    adapter_module = importlib.util.module_from_spec(adapter_spec)
    adapter_spec.loader.exec_module(adapter_module)
    manifest["optional_planner_sha256"] = hashlib.sha256(adapter_path.read_bytes()).hexdigest()
    Path("agent_source_manifest.json").write_text(json.dumps(manifest, indent=2))
    chooser = adapter_module.OllamaPlanner(MODEL)
    llm_run = studio.run("toy_good", proposer=chooser, max_steps=8)
    Path("local_planner_trace.json").write_text(json.dumps(chooser.trace, indent=2))
    print(llm_run["status"], llm_run["reason"])
else:
    print("Optional LLM planner disabled; no model request made.")
